# Do the placement labels hold on the study's own worlds?

**Question.** Sensors are chosen, and labelled, on a probe stack: K worlds at the study's operating point, simulated
with the probe seed (42), each converting one district. The study's worlds are simulated with another seed (43) and
were never seen by the selection. With `drift_all_districts`, a partition's worlds are again one world per drifting
district at one operating point — a stack in their own right. Running the same exact layer on them gives the exact
label of every link × drifting district on worlds the sensors were not chosen on. Three checks follow:

| § | question |
|---|---|
| labels | does each cell's exact quantity (S, N_r, g_w, Δm_k, σ) on the target worlds match the probe label? |
| classes | does the channel class of each cell (taxonomy) stay the same? |
| selection | would the rule pick the same sensors on the target worlds? |

This notebook replaced the pipeline's `verify_sensors` node and its `verification.csv` (Oct 2026), which could not
answer the question: it compared the exact probe label with the old settled-vs-init gain on one world (different
quantities, no counterfactual), and its `responded` flag fired on foreign branches whose response is exactly zero by
physics.

Reads the built worlds (`data/09_experiments/worlds/<sim_hash>/clone`), assembles one target stack per partition under
`notebooks/outputs/placement_generalisation/target_stacks/` (hard links to the world series where the volume allows,
copies otherwise) and runs `exact.ensure_exact` on it (cached; ~10 min per D-Town partition the first time).
Outputs: `label_cells.parquet` (every live link × k, probe vs target), `selection_stability.parquet`.

### Definitions

Both sides are the exact layer (`fedwater.placement.exact`), each on its own stack's windows (baseline, and drifting
district $k$'s settled window), paired against its own spliced no-drift city:

| quantity | definition |
|---|---|
| $S_{s,k}$ | shape effect: rms over the week of $z(PQ^{(k)}) - z(PQ^{(0)})$ (sd of the weekly profile) — the dependence label, per-client scaling |
| $N_{r,s,k}$ | $\lvert g_{w,s,k}\rvert / \lvert g_{w,s,\text{home}}\rvert$ — native gain relative to the home district's (shared scaling) |
| $g_{w,s,k}$ | native gain on the weekly profile, oriented: extra L/s through $s$ per extra L/s of $k$'s demand |
| $\Delta m_{s,k}$ | change of $k$'s share of $s$'s district water, drift world − no-drift city (the selection criterion) |
| $\sigma_{s,k}$ | storage share $\lvert g_{ie0}\rvert/(\lvert g_{de0}\rvert+\lvert g_{ie0}\rvert)$ (E2 twin with the hub on the no-drift trajectory); undefined where $\lvert g_{de0}\rvert+\lvert g_{ie0}\rvert <$ `exact.sigma_floor` (a ratio of two near-zero gains) |
| class | channel taxonomy of the cell (`select.taxonomy`, by $S$) |

Agreement per metric: Spearman $\rho$ across cells and the median $\lvert$probe − target$\rvert$; for thresholds, the
share of cells on the same side ($S\ge$ `S_floor`, $\lvert\Delta m\rvert\ge$ `dm_min`, σ channel demand / both / storage).
Cells are split into **own** (link in $k$'s district: large by construction) and **foreign** (the dependence cells);
scopes: the **placed** sensors and **all candidates** (the live links the rule chooses from). The seeds differ in the
demand draws only: same network, partition, operating point, transitions and horizon rule. What the probe drift and the
world drift share beyond that (seed node, front) is part of what this tests.

In [ ]:
%matplotlib inline
import warnings, logging, sys, pathlib, json, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.stats import spearmanr

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from fedwater.config import load_base_params
from fedwater.hashing import canonical_hash
from fedwater.experiments.spec import expand_study, load_studies
from fedwater.placement import exact as ex
from fedwater.placement import select as sel

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)
pd.set_option("display.width", 230, "display.max_columns", 60, "display.max_rows", 300)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})

STUDIES = ["dtown_districting_placement"]   # studies whose worlds were built (drift_all_districts: one world per district)
WORLDS = None                   # or {(network, partition): [sim_hash, ...]} to bypass the study expansion
FORCE_EXACT = False

BASE = load_base_params(REPO)
EXP_ROOT = REPO / load_studies(REPO).get("root", "data/09_experiments")
OUT = REPO / "notebooks" / "outputs" / "placement_generalisation"
TARGET_STORE = OUT / "target_stacks"           # assembled target stacks (hard links to the world files)
OUT.mkdir(parents=True, exist_ok=True)
RULE = {**sel.RULE_DEFAULTS, **(BASE["sensor_placement"].get("selection") or {})}
TAX = {**sel.TAXONOMY_DEFAULTS, **(BASE["sensor_placement"].get("taxonomy") or {})}
SLOTS = BASE["sensor_placement"]["slots"]
EXACT_CFG = BASE["sensor_placement"].get("exact") or {}
METRICS = {"S": "shape effect S", "Nr": "native gain / home", "g_w": "native gain g_w", "dm_k": "mixture change Δm_k",
           "sigma": "storage share σ"}
print(f"experiments {EXP_ROOT}  ->  {OUT}")

In [ ]:
# the built target worlds, grouped by (network, partition): one world per drifting district
if WORLDS is None:
    WORLDS = {}
    for st in STUDIES:
        for w in expand_study(st, REPO)["worlds"]:
            WORLDS.setdefault((w["network"], w["flat"]["districting"]), []).append(w["sim_hash"])
rows, GROUPS = [], {}
for key, hashes in WORLDS.items():
    for h in sorted(set(hashes)):
        mf = json.loads((EXP_ROOT / "worlds" / h / "manifest.json").read_text())
        rows.append({"network": key[0], "partition": key[1], "world": h, "status": mf["status"],
                     "drift": mf["world"].get("drift_district"), "sim_seed": mf["world"].get("sim_seed"),
                     "n_months": mf["world"].get("n_months")})
        if mf["status"] == "ok":
            GROUPS.setdefault(key, []).append((h, mf))
display(pd.DataFrame(rows))
for key, ws in list(GROUPS.items()):
    drifts = [m["world"]["drift_district"] for _, m in ws]
    if len(set(drifts)) != len(drifts) or len(drifts) < 2:
        display(Markdown(f"**{key}: skipped** -- needs one ok world per drifting district, got {drifts}"))
        GROUPS.pop(key)

## Target stacks and their exact layer

Gates as in the sweep notebook (G-X0 … G-X6c). G-X1 is the key one here: the target worlds can only be spliced into a
no-drift city if every non-drifting district's demand is the same series in every world of the partition.

In [ ]:
# every partition's target worlds assembled as a stack (the layout store.ensure_stack writes), then its exact layer;
# the probe stack's exact layer (the LABEL stack the placement's labels came from) alongside
def _link(src, dst):
    """Hard link when the volume allows it (no copy of the series), else a copy."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def target_stack(key, worlds):
    h0, m0 = worlds[0]
    clone0 = EXP_ROOT / "worlds" / h0 / "clone"
    ptr = yaml.safe_load((clone0 / "data/08_reporting/sensor_placement/stacks.yml").read_text())
    probe_path = Path(ptr["label"]["path"])
    probe_spec = json.loads((probe_path / "manifest.json").read_text())["spec"]
    hashes = sorted(h for h, _ in worlds)
    spec = {**probe_spec, "target_worlds": hashes}          # same windows rule (probe.settle, plan) as the probes
    sid = canonical_hash(spec)
    path = TARGET_STORE / key[0] / key[1] / sid
    (path / "worlds").mkdir(parents=True, exist_ok=True)
    part = yaml.safe_load((clone0 / "data/01_raw" / key[0] / "partitions" / key[1] / "districts.yml").read_text())
    (path / "districts.yml").write_text(yaml.safe_dump({"districts": part["districts"],
                                                        "assets": part.get("assets") or {}}, sort_keys=False))
    for h, mf in worlds:
        c = EXP_ROOT / "worlds" / h / "clone"
        d = mf["world"]["drift_district"]
        w = path / "worlds" / d
        w.mkdir(exist_ok=True)
        _link(c / "data/02_intermediate/demand_series.parquet", w / "demand.parquet")
        _link(c / "data/02_intermediate/flows.parquet", w / "flows.parquet")
        _link(c / "data/02_intermediate/wn_variant.pkl", w / "wn.pkl")
        shutil.copy2(c / "data/03_primary/gt_drift_schedule.csv", w / "schedule.csv")
        eff = mf["effective"]
        scen = json.loads(json.dumps(eff["scenario"]))
        scen.setdefault("drift", {})["tgt_district"] = d
        (w / "params.yml").write_text(yaml.safe_dump({"time": eff["time"], "scenario": scen,
                                                      "patterns": eff["patterns"]}, sort_keys=False))
    (path / "manifest.json").write_text(json.dumps({
        "status": "ok", "stack_hash": sid, "config_id": f"{key[1]}__target__{sid}", "network": key[0],
        "method": key[1], "districts": list(part["districts"]), "spec": spec, "target_worlds": hashes}, indent=1))
    return path, Path(ptr["label"]["exact"]), ptr


XT, XP, PTR, GT = {}, {}, {}, []
for key, ws in GROUPS.items():
    path, probe_exact, ptr = target_stack(key, ws)
    XT[key] = ex.ensure_exact(path, EXACT_CFG, force=FORCE_EXACT)
    XP[key] = ex.load_exact(probe_exact, EXACT_CFG)
    PTR[key] = ptr
    GT.append(XT[key]["gates"].assign(network=key[0], partition=key[1]))
GT = pd.concat(GT, ignore_index=True)
display(Markdown("**Exact-layer gates on the target worlds**"))
display(GT.pivot_table(index=["gate", "check"], columns=["network", "partition"], values="value", aggfunc="first"))
GATES_OK = bool(GT["passed"].all())
display(Markdown(f"**Gates {'pass' if GATES_OK else 'FAIL — do not read the comparison below'}** "
                 f"({int(GT['passed'].sum())}/{len(GT)})"))
display(Markdown("**Windows** (settled window of each drifting district, months: probes vs target worlds)"))
wrows = []
for key in GROUPS:
    sm = XT[key]["manifest"]["steps_month"]
    for k, wt in XT[key]["manifest"]["windows"].items():
        wp = XP[key]["manifest"]["windows"].get(k, {})
        wrows.append({"network": key[0], "partition": key[1], "k": k,
                      "probe settled": tuple(np.array(wp.get("settled", [np.nan, np.nan])) // sm),
                      "target settled": tuple(np.array(wt["settled"]) // sm)})
display(pd.DataFrame(wrows))

## Labels: probe vs target worlds

In [ ]:
# every (link, drifting district k) cell: probe-stack label (seed of the probes) vs the same exact quantity on the
# target worlds (the study's seed); scopes: the placed sensors, and every live candidate of the partition
CELLS, PLACED = [], {}
for key in GROUPS:
    h0 = GROUPS[key][0][0]
    pl = pd.read_csv(EXP_ROOT / "worlds" / h0 / "clone/data/03_primary/sensor_placement.csv", dtype={"element": str})
    PLACED[key] = pl
    LP, LT = XP[key]["labels"].copy(), XT[key]["labels"].copy()
    for L_ in (LP, LT):
        L_["element"] = L_["element"].astype(str)
        L_["cls"] = sel.taxonomy(L_, TAX, "S")
    keep = ["element", "k", "own"] + list(METRICS) + ["cls"]
    m = LP[keep].merge(LT[keep].drop(columns="own"), on=["element", "k"], suffixes=("_probe", "_target"))
    live = sel.candidates(sel.prep(XP[key], TAX), RULE)
    links = XP[key]["links"].set_index(XP[key]["links"]["element"].astype(str))
    fam = pl.set_index("element")["slot_class"]
    m = m[m["element"].isin(live.index)].assign(
        network=key[0], partition=key[1], home=lambda d: d["element"].map(links["home"]),
        topo=lambda d: d["element"].map(links["topo"]), placed=lambda d: d["element"].isin(set(pl["element"])),
        family=lambda d: d["element"].map(fam))
    CELLS.append(m)
CELLS = pd.concat(CELLS, ignore_index=True)
CELLS.to_parquet(OUT / "label_cells.parquet", index=False)

DM_MIN, S_FLOOR = float(RULE["dm_min"]), float(TAX["S_floor"])
CH = lambda s: np.select([s < TAX["ch_lo"], s > TAX["ch_hi"], np.isfinite(s)], ["demand", "storage", "both"], "n/a")


def agreement(g):
    out = {"cells": len(g)}
    for mt in METRICS:
        a, b = g[f"{mt}_probe"].to_numpy(float), g[f"{mt}_target"].to_numpy(float)
        ok = np.isfinite(a) & np.isfinite(b)
        out[f"rho {mt}"] = spearmanr(a[ok], b[ok]).correlation if ok.sum() > 3 and np.ptp(a[ok]) > 0 else np.nan
        out[f"med|Δ| {mt}"] = float(np.median(np.abs(a[ok] - b[ok]))) if ok.any() else np.nan
    out["same: S responds"] = float(((g["S_probe"] >= S_FLOOR) == (g["S_target"] >= S_FLOOR)).mean())
    out["same: |Δm| >= dm_min"] = float(((g["dm_k_probe"].abs() >= DM_MIN) == (g["dm_k_target"].abs() >= DM_MIN)).mean())
    sp, st = CH(g["sigma_probe"].to_numpy(float)), CH(g["sigma_target"].to_numpy(float))
    both = (sp != "n/a") & (st != "n/a")
    out["same: σ channel"] = float((sp[both] == st[both]).mean()) if both.any() else np.nan
    out["same: class"] = float((g["cls_probe"] == g["cls_target"]).mean())
    return pd.Series(out)


TAB = []
for scope, sub in (("placed", CELLS[CELLS["placed"]]), ("all candidates", CELLS)):
    for cell, g in (("own", sub[sub["own"]]), ("foreign", sub[~sub["own"]])):
        for key, gg in g.groupby(["network", "partition"]):
            TAB.append(agreement(gg).rename((*key, scope, cell)))
TAB = pd.DataFrame(TAB)
TAB.index = pd.MultiIndex.from_tuples(TAB.index, names=["network", "partition", "scope", "cell"])
display(Markdown("**Agreement, probe label vs target worlds** — rank correlation and median absolute difference per "
                 "metric; share of cells on the same side of each threshold"))
display(TAB.round(3))

### Cell by cell

Foreign cells only (own cells sit far up the diagonal and would compress the axes). Grey: every live candidate;
blue / red: the placed branches / transits. On the dotted line the label is reproduced exactly.

In [ ]:
# probe label (x) vs target worlds (y), foreign cells: every live candidate (grey), placed branches and transits
FAM_C = {"branch": "#08519c", "transit": "#a50f15"}
for key in GROUPS:
    g = CELLS[(CELLS["network"] == key[0]) & (CELLS["partition"] == key[1]) & ~CELLS["own"]]
    fig, axes = plt.subplots(1, len(METRICS), figsize=(4.6 * len(METRICS), 4.6), squeeze=False)
    for j, (mt, lab) in enumerate(METRICS.items()):
        ax = axes[0, j]
        a, b = g[f"{mt}_probe"].astype(float), g[f"{mt}_target"].astype(float)
        ax.scatter(a[~g["placed"]], b[~g["placed"]], s=5, color="0.75", label="candidates", zorder=1)
        for f_, c_ in FAM_C.items():
            s_ = g["family"] == f_
            ax.scatter(a[s_], b[s_], s=28, color=c_, edgecolor="w", lw=0.5, label=f"placed {f_}", zorder=3)
        lo = np.nanmin([a.min(), b.min()]) if len(g) else 0
        hi = np.nanmax([a.max(), b.max()]) if len(g) else 1
        ax.plot([lo, hi], [lo, hi], "k:", lw=0.8, zorder=2)
        ax.set(xlabel=f"probe stack ({lab})", ylabel="target worlds", title=lab)
        if mt == "S":
            ax.axhline(S_FLOOR, color="0.5", lw=0.6); ax.axvline(S_FLOOR, color="0.5", lw=0.6)
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(f"{key[0]} / {key[1]} — foreign cells (link × drifting district not its home): probe label vs the "
                 "same exact quantity on the study's worlds")
    fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

# the placed sensors, cell by cell
for key in GROUPS:
    g = CELLS[(CELLS["network"] == key[0]) & (CELLS["partition"] == key[1]) & CELLS["placed"] & ~CELLS["own"]]
    pl = PLACED[key].set_index("element")
    t = g.assign(client=g["element"].map(pl["district"]), slot=g["element"].map(pl["slot"]))
    cols = ["client", "slot", "element", "k", "S_probe", "S_target", "dm_k_probe", "dm_k_target", "sigma_probe",
            "sigma_target", "cls_probe", "cls_target"]
    display(Markdown(f"**{key[0]} / {key[1]} — placed transits, foreign cells**"))
    display(t[t["family"] == "transit"][cols].sort_values(["client", "slot", "k"]).round(3))

## Selection stability

The pipeline's rule (`placement.select`, shipped settings) re-run on the target worlds' exact layer. A slot that
changes link is not necessarily a worse sensor (links in series carry the same water); the cell tables above say whether
what the placed sensors carry and see holds.

In [ ]:
# would the rule choose the same sensors on the target worlds' exact layer?
SEL = []
for key in GROUPS:
    pl = PLACED[key]
    names = list(XT[key]["manifest"]["districts"])
    try:
        picks, log = sel.select_slots(sel.prep(XT[key], TAX), names, SLOTS, RULE, TAX)
        note = ""
    except sel.InsufficientCandidates as exc:
        picks, log = sel.select_flow(sel.prep(XT[key], TAX), RULE, TAX,
                                     int(SLOTS["flow"]["branch"]), int(SLOTS["flow"]["transit"]))
        note = f"short: {exc}"[:200]
    a = pl.set_index(["district", "slot"])["element"]
    b = picks.set_index(["district", "slot"])["element"]
    for d in names:
        pa, pb = a.xs(d, level=0), b.xs(d, level=0) if d in b.index.get_level_values(0) else pd.Series(dtype=str)
        SEL.append({"network": key[0], "partition": key[1], "client": d,
                    "same slot→link": int(sum(pa.get(s) == pb.get(s) for s in pa.index)), "slots": len(pa),
                    "same set": len(set(pa) & set(pb)),
                    "placed (probes)": ", ".join(f"{s.split('_', 1)[1]}:{e}" for s, e in pa.items()),
                    "target-worlds pick": ", ".join(f"{s.split('_', 1)[1]}:{e}" for s, e in pb.items()), "note": note})
SEL = pd.DataFrame(SEL)
SEL.to_parquet(OUT / "selection_stability.parquet", index=False)
display(Markdown("**Selection stability** — the rule re-run on the target worlds' exact layer vs the placement chosen "
                 "on the probe stack"))
display(SEL.set_index(["network", "partition", "client"]))